<a href="https://colab.research.google.com/github/oliviaguo1234/OliviaGuo_TCS/blob/main/OliviaGuo_loans.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [61]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
from sklearn.feature_selection import SelectKBest
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error, accuracy_score

In [62]:
import pandas as pd
import numpy as np

loans_df = pd.read_csv("lendingclub_loans_export.csv")

#Change NA values to null
loans_df = loans_df.replace("NA", np.nan)

#Droped join annual income columns (>85% null rate for both)
loans_df.drop(columns = ["annual_income_joint", "verification_income_joint", "debt_to_income_joint"], inplace = True)

#All loans were 2018, remove year from issue month
loans_df["issue_month"] = pd.to_datetime(loans_df["issue_month"], format="%b-%Y").dt.strftime

#Grade information included in subgrade
loans_df.drop(columns = ["grade"], inplace = True)

loans_df.to_csv("lendingclub_loans_clean.csv")

In [64]:
loans_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 51 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   emp_title                         9167 non-null   object 
 1   emp_length                        9183 non-null   float64
 2   state                             10000 non-null  object 
 3   homeownership                     10000 non-null  object 
 4   annual_income                     10000 non-null  float64
 5   verified_income                   10000 non-null  object 
 6   debt_to_income                    9976 non-null   float64
 7   delinq_2y                         10000 non-null  int64  
 8   months_since_last_delinq          4342 non-null   float64
 9   earliest_credit_line              10000 non-null  int64  
 10  inquiries_last_12m                10000 non-null  int64  
 11  total_credit_lines                10000 non-null  int64  
 12  open_

In [65]:
#Drop debt_to_income
loans_df = loans_df.dropna(how='any', subset=['debt_to_income']).copy()
#Replace months_since_last_delinq missing values with 0
loans_df['months_since_last_delinq'] = loans_df['months_since_last_delinq'].fillna(value=0)
#Replace null emp_title with "unknown"
loans_df['emp_title'] = loans_df['emp_title'].fillna(value = 'unknown')
#Replace num_accounts_120d_past_due missing values with 0
loans_df['num_accounts_120d_past_due'] = loans_df['num_accounts_120d_past_due'].fillna(value=0)
#Replace emp_length missing values with 0
loans_df['emp_length'] = loans_df['emp_length'].fillna(value=0)
loans_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9976 entries, 0 to 9999
Data columns (total 51 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   emp_title                         9976 non-null   object 
 1   emp_length                        9976 non-null   float64
 2   state                             9976 non-null   object 
 3   homeownership                     9976 non-null   object 
 4   annual_income                     9976 non-null   float64
 5   verified_income                   9976 non-null   object 
 6   debt_to_income                    9976 non-null   float64
 7   delinq_2y                         9976 non-null   int64  
 8   months_since_last_delinq          9976 non-null   float64
 9   earliest_credit_line              9976 non-null   int64  
 10  inquiries_last_12m                9976 non-null   int64  
 11  total_credit_lines                9976 non-null   int64  
 12  open_credit

In [66]:
#create separate column for months_since_90d_late called months_since_90d_relevance
loans_df['months_since_90d_relevance'] = loans_df['months_since_90d_late'].notnull()


#create separate column for months_since_last_credit_inquiry called months_since_credit_relevance
loans_df['months_since_credit_relevance'] = loans_df['months_since_last_credit_inquiry'].notnull()

In [67]:
loans_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9976 entries, 0 to 9999
Data columns (total 53 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   emp_title                         9976 non-null   object 
 1   emp_length                        9976 non-null   float64
 2   state                             9976 non-null   object 
 3   homeownership                     9976 non-null   object 
 4   annual_income                     9976 non-null   float64
 5   verified_income                   9976 non-null   object 
 6   debt_to_income                    9976 non-null   float64
 7   delinq_2y                         9976 non-null   int64  
 8   months_since_last_delinq          9976 non-null   float64
 9   earliest_credit_line              9976 non-null   int64  
 10  inquiries_last_12m                9976 non-null   int64  
 11  total_credit_lines                9976 non-null   int64  
 12  open_credit